# 05. 전체 산출물 검증·보고서

앞의 네 노트북이 만든 JSON·NPY·manifest를 서로 대조합니다. 모든 검증이 통과해야 최종 보고서를 저장합니다. ChromaDB 적재는 수행하지 않습니다.

In [1]:
from pathlib import Path
from collections import Counter
import hashlib
import json
import os
import tempfile

import numpy as np
import pandas as pd
from IPython.display import display
from transformers import AutoTokenizer

def find_data_dir():
    for candidate in [Path.cwd(), Path.cwd() / 'ㅋㅌㅊ', Path.cwd().parent, Path.cwd().parent / 'ㅋㅌㅊ']:
        resolved = candidate.resolve()
        if (resolved / 'output/RAG/maple_inven_tips_embeddings_manifest.json').is_file():
            return resolved
    raise FileNotFoundError('04_embedding.ipynb를 먼저 실행하세요.')

DATA_DIR = find_data_dir()
OUTPUT_ROOT = DATA_DIR / 'output'
RAW_PATH = OUTPUT_ROOT / 'intermediate/maple_inven_tips_raw.json'
SETTINGS_PATH = OUTPUT_ROOT / 'intermediate/pipeline_settings.json'
PROCESSED_PATH = OUTPUT_ROOT / 'processed/maple_inven_tips_processed.json'
REJECTED_PATH = OUTPUT_ROOT / 'processed/maple_inven_tips_rejected.json'
CHUNKS_PATH = OUTPUT_ROOT / 'RAG/maple_inven_tips_documents_chunked.json'
EMBEDDINGS_PATH = OUTPUT_ROOT / 'RAG/maple_inven_tips_embeddings.npy'
MANIFEST_PATH = OUTPUT_ROOT / 'RAG/maple_inven_tips_embeddings_manifest.json'
REPORT_PATH = OUTPUT_ROOT / 'RAG/maple_inven_tips_pipeline_report.json'

print('검증 대상:', OUTPUT_ROOT)

C:\Users\Playdata\Desktop\team3_ 프로젝트1\mle-01-p1-team3\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


검증 대상: C:\Users\Playdata\Desktop\team3_ 프로젝트1\mle-01-p1-team3\ㅋㅌㅊ\output


In [2]:
def read_json(path):
    return json.loads(path.read_text(encoding='utf-8'))

def sha256_file(path):
    digest = hashlib.sha256()
    with path.open('rb') as stream:
        for block in iter(lambda: stream.read(1024 * 1024), b''):
            digest.update(block)
    return digest.hexdigest()

def atomic_write_json(path, value):
    path.parent.mkdir(parents=True, exist_ok=True)
    temporary = None
    try:
        with tempfile.NamedTemporaryFile('w', encoding='utf-8', newline='', dir=path.parent, delete=False) as stream:
            json.dump(value, stream, ensure_ascii=False, indent=2)
            stream.write('\n')
            temporary = Path(stream.name)
        os.replace(temporary, path)
    finally:
        if temporary is not None and temporary.exists():
            temporary.unlink()

In [3]:
settings = read_json(SETTINGS_PATH)
raw_rows = read_json(RAW_PATH)
processed = read_json(PROCESSED_PATH)
rejected = read_json(REJECTED_PATH)
chunks = read_json(CHUNKS_PATH)
manifest = read_json(MANIFEST_PATH)
vectors = np.load(EMBEDDINGS_PATH, allow_pickle=False)
chunk_ids = [chunk['id'] for chunk in chunks]
norms = np.linalg.norm(vectors, axis=1)

tokenizer = AutoTokenizer.from_pretrained(settings['model_name'])
tokenizer.model_max_length = 10**9
token_counts = [
    len(tokenizer.encode(
        f"{chunk['metadata']['embedding_prefix']}\n\n{chunk['page_content']}",
        add_special_tokens=True, truncation=False,
    ))
    for chunk in chunks
]

checks = {
    '입력 300행': len(raw_rows) == 300,
    '유효 문서 299건': len(processed) == 299,
    '제외 1건': len(rejected) == 1,
    '청크·벡터 행 수 일치': len(chunks) == vectors.shape[0] == manifest['embedding_count'],
    '청크 ID 고유': len(chunk_ids) == len(set(chunk_ids)),
    '청크 ID 순서 일치': chunk_ids == manifest['chunk_ids'],
    'float32': vectors.dtype == np.float32,
    '모든 벡터 유한값': bool(np.isfinite(vectors).all()),
    'L2 정규화': bool(np.allclose(norms, 1.0, atol=1e-5)),
    '128토큰 이하': max(token_counts) <= settings['max_tokens'],
    '청크 JSON checksum': sha256_file(CHUNKS_PATH) == manifest['chunks_sha256'],
    '임베딩 NPY checksum': sha256_file(EMBEDDINGS_PATH) == manifest['embeddings_sha256'],
}
failed = [name for name, passed in checks.items() if not passed]
if failed:
    raise AssertionError(f"검증 실패: {', '.join(failed)}")
display(pd.DataFrame({'검사항목': list(checks), '통과': list(checks.values())}))

,검사항목,통과
0,입력 300행,True
1,유효 문서 299건,True
2,제외 1건,True
3,청크·벡터 행 수 일치,True
4,청크 ID 고유,True
5,청크 ID 순서 일치,True
6,float32,True
7,모든 벡터 유한값,True
8,L2 정규화,True
9,128토큰 이하,True


In [4]:
report = {
    'input_rows': len(raw_rows),
    'accepted_rows': len(processed),
    'rejected_rows': len(rejected),
    'duplicate_rows': sum(item['reason'] == 'duplicate_url_replaced' for item in rejected),
    'input_files': settings['input_files'],
    'chunk_count': len(chunks),
    'embedding_count': int(vectors.shape[0]),
    'embedding_dimension': int(vectors.shape[1]),
    'categories': dict(sorted(Counter(item['category'] for item in processed).items())),
    'short_text_rows': sum(item['text_quality'] == 'short' for item in processed),
    'min_input_tokens': min(token_counts),
    'max_input_tokens': max(token_counts),
    'validation_passed': True,
}
atomic_write_json(REPORT_PATH, report)
display(report)

artifact_paths = [PROCESSED_PATH, REJECTED_PATH, CHUNKS_PATH, EMBEDDINGS_PATH, MANIFEST_PATH, REPORT_PATH]
display(pd.DataFrame([
    {'파일': str(path.relative_to(DATA_DIR)), '크기(MB)': round(path.stat().st_size / 1024**2, 3)}
    for path in artifact_paths
]))
print('모든 검증 통과 — ChromaDB 적재 직전 산출물 생성 완료')

{'input_rows': 300,
 'accepted_rows': 299,
 'rejected_rows': 1,
 'duplicate_rows': 0,
 'input_files': ['C:\\Users\\Playdata\\Desktop\\team3_ 프로젝트1\\mle-01-p1-team3\\ㅋㅌㅊ\\maple_inven_rag_원본(1~10p).csv'],
 'chunk_count': 5506,
 'embedding_count': 5506,
 'embedding_dimension': 768,
 'categories': {'기타': 154,
  '리부트': 1,
  '메이플M': 5,
  '몬스터': 29,
  '사냥': 27,
  '수다': 2,
  '실험': 9,
  '아이템': 24,
  '전문기술': 7,
  '지역': 2,
  '퀘스트': 21,
  '테섭': 18},
 'short_text_rows': 19,
 'min_input_tokens': 21,
 'max_input_tokens': 128,
 'validation_passed': True}

,파일,크기(MB)
0,output\processed\maple_inven_tips_processed.json,1.879
1,output\processed\maple_inven_tips_rejected.json,0.000
2,output\RAG\maple_inven_tips_documents_chunked....,5.856
3,output\RAG\maple_inven_tips_embeddings.npy,16.131
4,output\RAG\maple_inven_tips_embeddings_manifes...,0.135
5,output\RAG\maple_inven_tips_pipeline_report.json,0.001


모든 검증 통과 — ChromaDB 적재 직전 산출물 생성 완료
